# polars_reg Showcase

A tour of everything `polars_reg` can do: OLS, fixed effects, instrumental variables, panel estimators, robust/clustered standard errors, pretty-printed tables, and Stata equivalence checking.

```
pip install polars_reg
```

In [ ]:
import numpy as np
import polars as pl
import polars_reg as pr

# Reproducible data
rng = np.random.default_rng(42)

## 1. Generate sample data

A simulated panel dataset with firm and industry fixed effects, an endogenous variable, and instruments.

In [ ]:
n_firms, n_years = 100, 10
n = n_firms * n_years

firm_id = np.repeat(np.arange(n_firms), n_years)
year_id = np.tile(np.arange(2010, 2010 + n_years), n_firms)
industry = np.repeat(rng.choice(["Tech", "Finance", "Health", "Energy"], n_firms), n_years)

# Fixed effects
firm_fe = rng.standard_normal(n_firms)
year_fe = rng.standard_normal(n_years) * 0.5

# Regressors
x1 = rng.standard_normal(n)
x2 = rng.standard_normal(n)

# Endogenous variable + instruments
z1 = rng.standard_normal(n)
z2 = rng.standard_normal(n)
u = rng.standard_normal(n)  # correlated error
x_endog = 0.5 * z1 + 0.3 * z2 + 0.8 * u

# Outcome
y = 2.0 + 1.0 * x1 - 0.5 * x2 + 1.5 * x_endog + firm_fe[firm_id] + year_fe[year_id % n_years] + u

df = pl.DataFrame({
    "y": y, "x1": x1, "x2": x2,
    "x_endog": x_endog, "z1": z1, "z2": z2,
    "firm_id": firm_id, "year": year_id, "industry": industry,
})
df.head()

---
## 2. OLS — Basic, Robust, and Clustered Standard Errors

In [ ]:
# Basic OLS with iid standard errors
r_iid = pr.ols("y ~ x1 + x2 + x_endog", data=df)
print(r_iid.summary())

In [ ]:
# Robust standard errors (HC1, HC2, HC3)
r_robust = pr.ols("y ~ x1 + x2 + x_endog", data=df, vcov="HC1")
print(r_robust.summary())

In [ ]:
# Clustered standard errors (one-way)
r_cl1 = pr.ols("y ~ x1 + x2 + x_endog", data=df, cluster="firm_id")
print(r_cl1.summary())

In [ ]:
# Multi-way clustered standard errors (Cameron-Gelbach-Miller)
r_cl2 = pr.ols("y ~ x1 + x2 + x_endog", data=df, cluster=["firm_id", "year"])
print(r_cl2.summary())

---
## 3. High-Dimensional Fixed Effects (reghdfe-style)

Absorb firm and/or year fixed effects via iterative demeaning. The `|` separator in the formula specifies which variables to absorb.

In [ ]:
# One-way FE: absorb firm
r_fe1 = pr.ols("y ~ x1 + x2 + x_endog | firm_id", data=df, cluster="firm_id")
print(r_fe1.summary())

In [ ]:
# Two-way FE: absorb firm + year, two-way clustered SEs
r_fe2 = pr.ols("y ~ x1 + x2 + x_endog | firm_id + year", data=df, cluster=["firm_id", "year"])
print(r_fe2.summary())

In [ ]:
# Three-way FE: firm + year + industry
r_fe3 = pr.ols("y ~ x1 + x2 | firm_id + year + industry", data=df, cluster="firm_id")
print(r_fe3.summary())

---
## 4. Instrumental Variables

Use `||` to separate the IV equation: `y ~ exog_vars || endog_var ~ instruments`

With fixed effects: `y ~ exog_vars | fe_vars | endog_var ~ instruments`

In [ ]:
# 2SLS: instrument x_endog with z1 and z2
r_2sls = pr.iv2sls("y ~ x1 + x2 || x_endog ~ z1 + z2", data=df)
print(r_2sls.summary())

In [ ]:
# 2SLS with absorbed firm FE
r_2sls_fe = pr.iv2sls("y ~ x1 + x2 | firm_id | x_endog ~ z1 + z2", data=df)
print(r_2sls_fe.summary())

In [ ]:
# LIML — more robust to weak instruments than 2SLS
r_liml = pr.liml("y ~ x1 + x2 || x_endog ~ z1 + z2", data=df)
print(r_liml.summary())

In [ ]:
# GMM-IV — efficient two-step GMM with Hansen J overidentification test
r_gmm = pr.gmm_iv("y ~ x1 + x2 || x_endog ~ z1 + z2", data=df)
print(r_gmm.summary())

---
## 5. Panel Estimators

Fixed effects (within), random effects (Swamy-Arora GLS), and first-difference.

In [ ]:
# Panel fixed effects (within estimator, SEs clustered by entity)
r_pfe = pr.panel_fe("y ~ x1 + x2 + x_endog", data=df, entity="firm_id", time="year")
print(r_pfe.summary())

In [ ]:
# Panel random effects (GLS with Swamy-Arora variance components)
r_pre = pr.panel_re("y ~ x1 + x2 + x_endog", data=df, entity="firm_id")
print(r_pre.summary())

In [ ]:
# Panel first-difference
r_pfd = pr.panel_fd("y ~ x1 + x2 + x_endog", data=df, entity="firm_id", time="year")
print(r_pfd.summary())

---
## 6. Result Object — Programmatic Access

Every regression returns a `RegressionResult` with coefficients, standard errors, t-stats, p-values, confidence intervals, coefficient tables, fitted values, and predictions.

In [ ]:
r = pr.ols("y ~ x1 + x2", data=df)

print("Coefficients:", r.coefficients)
print("Std errors:  ", r.se)
print("t-stats:     ", r.tstat)
print("p-values:    ", r.pvalue)
print("95% CI:\n", r.confint())
print("\nR²:", r.r_squared, " Adj R²:", r.r_squared_adj, " N:", r.n_obs)

In [ ]:
# Coefficient table as a Polars DataFrame — easy to filter, export, etc.
r.coef_table()

In [ ]:
# Fitted values and prediction
y_hat = r.fitted()
print("In-sample fitted values (first 5):", y_hat[:5])

# Out-of-sample prediction with new data
new_df = pl.DataFrame({"x1": [1.0, 2.0], "x2": [0.5, -0.5]})
print("Predictions:", r.predict(new_df))

---
## 7. regtable — Side-by-Side Regression Comparison (estout-style)

Compare multiple specifications in a single compact table. Each FE and cluster variable gets its own Y/N indicator row.

In [ ]:
# Build up a specification progressively
m1 = pr.ols("y ~ x1 + x2 + x_endog", data=df)
m2 = pr.ols("y ~ x1 + x2 + x_endog", data=df, vcov="HC1")
m3 = pr.ols("y ~ x1 + x2 + x_endog | firm_id", data=df, cluster="firm_id")
m4 = pr.ols("y ~ x1 + x2 + x_endog | firm_id + year", data=df, cluster=["firm_id", "year"])

pr.regtable(m1, m2, m3, m4, labels=["OLS", "Robust", "Firm FE", "Twoway FE"])

In [ ]:
# Compare IV estimators side by side
pr.regtable(r_2sls, r_liml, r_gmm, labels=["2SLS", "LIML", "GMM"])

In [ ]:
# Compare panel estimators
pr.regtable(r_pfe, r_pre, r_pfd, labels=["Panel FE", "Panel RE", "Panel FD"])

In [ ]:
# Customize: no stars, higher precision
pr.regtable(m1, m3, stars=False, precision=6)

---
## 8. GroupBy Regression — Run per Group

`groupby_reg()` runs the same regression for each group in the data — useful for estimating factor loadings per stock, running regressions per industry, etc.

In [ ]:
# Run OLS per industry group
grp = pr.groupby_reg(pr.ols, "y ~ x1 + x2 + x_endog", df, group_by="industry")
print(grp.summary())

In [ ]:
# Stacked coefficient table across all groups
grp.coef_table()

In [ ]:
# Use regtable to compare groups side by side
pr.regtable(*grp.values(), labels=list(grp.keys()))

---
## 9. Stata Equivalence — Generate Stata Code

`to_stata()` translates any `polars_reg` call into the equivalent Stata command, so you can verify results in Stata or share reproducible code with Stata users.

In [ ]:
# OLS → reg
print(pr.to_stata("ols", "y ~ x1 + x2"))
print()

# OLS with robust SEs → reg, vce(robust)
print(pr.to_stata("ols", "y ~ x1 + x2", vcov="HC1"))
print()

# reghdfe with multi-way FE and clustering
print(pr.to_stata("ols", "y ~ x1 + x2 | firm_id + year", cluster=["firm_id", "year"]))
print()

# 2SLS → ivregress 2sls
print(pr.to_stata("iv2sls", "y ~ x1 + x2 || x_endog ~ z1 + z2"))
print()

# 2SLS with FE → ivreghdfe
print(pr.to_stata("iv2sls", "y ~ x1 + x2 | firm_id | x_endog ~ z1 + z2"))
print()

# LIML
print(pr.to_stata("liml", "y ~ x1 + x2 || x_endog ~ z1 + z2"))
print()

# GMM
print(pr.to_stata("gmm_iv", "y ~ x1 + x2 || x_endog ~ z1 + z2"))
print()

# Panel FE → xtreg, fe
print(pr.to_stata("panel_fe", "y ~ x1 + x2", entity="firm_id", time="year"))
print()

# Panel RE → xtreg, re
print(pr.to_stata("panel_re", "y ~ x1 + x2", entity="firm_id"))
print()

# Panel FD → reg D.y D.x1 D.x2
print(pr.to_stata("panel_fd", "y ~ x1 + x2", entity="firm_id", time="year"))

In [ ]:
# Generate pystata-compatible Python code for automated comparison
print(pr.to_stata("ols", "y ~ x1 + x2 | firm_id", cluster=["firm_id"], pystata=True))

In [ ]:
# compare_stata() runs polars_reg and (if pystata is installed) Stata side by side
# Without pystata, it shows polars_reg results + the Stata command to run manually
report = pr.compare_stata("ols", "y ~ x1 + x2", data=df)

---
## 10. R Equivalence — Generate R Code

`to_r()` translates any `polars_reg` call into equivalent R code, mapping to `lm()`, `fixest::feols()`, `AER::ivreg()`, or `plm::plm()` as appropriate.

In [ ]:
# OLS → lm()
print(pr.to_r("ols", "y ~ x1 + x2"))
print()

# OLS with robust SEs → sandwich::vcovHC
print(pr.to_r("ols", "y ~ x1 + x2", vcov="HC1"))
print()

# OLS with clustering → fixest::feols
print(pr.to_r("ols", "y ~ x1 + x2", cluster=["firm_id"]))
print()

# reghdfe-style FE + clustering → fixest::feols
print(pr.to_r("ols", "y ~ x1 + x2 | firm_id + year", cluster=["firm_id", "year"]))
print()

# 2SLS → fixest::feols with IV syntax
print(pr.to_r("iv2sls", "y ~ x1 + x2 || x_endog ~ z1 + z2"))
print()

# 2SLS with FE → fixest::feols
print(pr.to_r("iv2sls", "y ~ x1 + x2 | firm_id | x_endog ~ z1 + z2"))
print()

# LIML → AER::ivreg
print(pr.to_r("liml", "y ~ x1 + x2 || x_endog ~ z1 + z2"))
print()

# GMM → advisory note + feols fallback
print(pr.to_r("gmm_iv", "y ~ x1 + x2 || x_endog ~ z1 + z2"))
print()

# Panel FE → plm with model="within"
print(pr.to_r("panel_fe", "y ~ x1 + x2", entity="firm_id", time="year"))
print()

# Panel RE → plm with model="random"
print(pr.to_r("panel_re", "y ~ x1 + x2", entity="firm_id"))
print()

# Panel FD → plm with model="fd"
print(pr.to_r("panel_fd", "y ~ x1 + x2", entity="firm_id", time="year"))

In [ ]:
# compare_r() runs polars_reg and (if rpy2 is installed) R side by side
# Without rpy2, it shows polars_reg results + the R code to run manually
report = pr.compare_r("ols", "y ~ x1 + x2", data=df)

---
## 11. Formula Syntax Reference

| polars_reg | Meaning | Stata | R |
|---|---|---|---|
| `y ~ x1 + x2` | OLS | `reg y x1 x2` | `lm(y ~ x1 + x2)` |
| `y ~ x1 + x2 - 1` | No intercept | `reg y x1 x2, noconstant` | `lm(y ~ x1 + x2 - 1)` |
| `y ~ x1 \| fe1` | Absorbed FE | `reghdfe y x1, absorb(fe1)` | `feols(y ~ x1 \| fe1)` |
| `y ~ x1 \| fe1 + fe2` | Multi-way FE | `reghdfe y x1, absorb(fe1 fe2)` | `feols(y ~ x1 \| fe1 + fe2)` |
| `y ~ x1 \|\| x_end ~ z1 + z2` | IV/2SLS | `ivregress 2sls y x1 (x_end = z1 z2)` | `ivreg(y ~ x1 + x2 \| x_end \| z1 + z2)` |
| `y ~ x1 \| fe1 \| x_end ~ z1` | IV + FE | `ivreghdfe y x1 (x_end = z1), absorb(fe1)` | `feols(y ~ x1 \| fe1 \| x_end ~ z1)` |

**R packages:** `fixest::feols()` for FE/clustering, `AER::ivreg()` for IV, base `lm()` for OLS. `fixest` uses the same `|` separator convention for fixed effects that polars_reg does.